In [1]:
import os
import pandas as pd

# 1. 定义输入与输出文件夹
dataset_dir = 'origin'
output_dir = 'merged_dataset'  # 存放 6 个最终合并文件的新文件夹

# 如果输出文件夹不存在，则自动创建
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. 定义遍历的年份与 Zone
years = [2016, 2017, 2018, 2019, 2020, 2021]
zones = [1, 2, 3, 4, 5]

# 3. 按年份合并数据（将生成 6 个独立循环）
for year in years:
    dfs_to_concat = []
    
    # 收集当前年份下所有 Zone 的数据
    for zone in zones:
        # 构建路径格式，如: dataset/zone1/Features_Zone1_2016.csv
        file_path = os.path.join(dataset_dir, f'zone{zone}', f'Academic_Zone{zone}_{year}.csv')
        
        if os.path.exists(file_path):
            try:
                df = pd.read_csv(file_path)
                # 可选：如果你想在合并后的表里知道这行数据原本属于哪个区，可以加下面这行代码
                # df['Original_Zone'] = zone 
                dfs_to_concat.append(df)
            except Exception as e:
                print(f"读取异常 {file_path}: {e}")
        else:
            print(f"提示: 未找到文件 {file_path}，已自动跳过。")
            
    # 当该年份收集到数据时，执行按行拼接并导出
    if dfs_to_concat:
        # ignore_index=True 确保行号连续，不会因为多个表合并导致 index 重复
        merged_df = pd.concat(dfs_to_concat, ignore_index=True)
        
        # 导出路径，如: merged_dataset/Merged_Features_2016.csv
        output_file = os.path.join(output_dir, f'Merged_Features_{year}.csv')
        
        # 导出 CSV，丢弃无意义的默认数字索引列
        merged_df.to_csv(output_file, index=False)
        print(f"成功: {year} 年数据已合并，输出路径 -> {output_file} (总行数: {len(merged_df)})")
    else:
        print(f"警告: {year} 年在所有子文件夹中均无有效数据。")

print("\n合并任务执行完毕，共导出最多 6 个 CSV 文件。")

成功: 2016 年数据已合并，输出路径 -> merged_dataset\Merged_Features_2016.csv (总行数: 19829)
成功: 2017 年数据已合并，输出路径 -> merged_dataset\Merged_Features_2017.csv (总行数: 19898)
成功: 2018 年数据已合并，输出路径 -> merged_dataset\Merged_Features_2018.csv (总行数: 19878)
成功: 2019 年数据已合并，输出路径 -> merged_dataset\Merged_Features_2019.csv (总行数: 19965)
提示: 未找到文件 origin\zone1\Academic_Zone1_2020.csv，已自动跳过。
成功: 2020 年数据已合并，输出路径 -> merged_dataset\Merged_Features_2020.csv (总行数: 19920)
提示: 未找到文件 origin\zone1\Academic_Zone1_2021.csv，已自动跳过。
成功: 2021 年数据已合并，输出路径 -> merged_dataset\Merged_Features_2021.csv (总行数: 19895)

合并任务执行完毕，共导出最多 6 个 CSV 文件。


In [2]:
import pandas as pd
import numpy as np
import os
import glob

def engineer_features(df):
    """
    基于无时间泄露原则，为冬小麦数据集添加各阶段的衍生特征。
    """
    periods = ['P1', 'P2', 'P3', 'P4', 'P5']
    
    # ---------------------------------------------------------
    # 1. 阶段内状态耦合特征 (所有阶段 P1-P5 均可计算)
    # ---------------------------------------------------------
    for p in periods:
        # 1.1 水分利用效率 (WUE) = NIRv / SM
        df[f'{p}_WUE'] = df[f'{p}_NIRv'] / (df[f'{p}_SM'] + 0.001)
        
        # 1.2 大气-土壤脱耦压力 = VPD_max / SM 
        df[f'{p}_Decoupling_Stress'] = df[f'{p}_VPD_max'] / (df[f'{p}_SM'] + 0.001)
        
        # 1.3 热量转化效率 = NDVI_max / GDD
        df[f'{p}_Thermal_Efficiency'] = df[f'{p}_NDVI_max'] / (df[f'{p}_GDD'] + 1.0)
        
        # 1.4 雨热平衡度 = PPT / Tmean (处理 Tmean <= 0 的情况，避免分母为负或零)
        tmean_safe = np.where(df[f'{p}_Tmean'] <= 0, 0.1, df[f'{p}_Tmean'])
        df[f'{p}_Hydrothermal_Balance'] = df[f'{p}_PPT'] / tmean_safe

    # ---------------------------------------------------------
    # 2. 静态-动态交叉特征 (所有阶段 P1-P5 均可计算)
    # ---------------------------------------------------------
    for p in periods:
        # 2.1 土壤保水脆弱度 = VPD / Clay
        df[f'{p}_Drought_Vulnerability'] = df[f'{p}_VPD'] / (df['Clay'] + 0.1)
        
        # 2.2 本底肥力驱动势 = NDVI * SOC
        df[f'{p}_Fertility_Vigor'] = df[f'{p}_NDVI'] * df['SOC']

    # ---------------------------------------------------------
    # 3. 历史状态累积特征 (按阶段逐步累加)
    # ---------------------------------------------------------
    metrics_to_accumulate = ['GDD', 'PPT', 'FDD', 'VPD']
    for metric in metrics_to_accumulate:
        current_sum = np.zeros(len(df))
        for p in periods:
            current_sum = current_sum + df[f'{p}_{metric}']
            df[f'Cum_{metric}_{p}'] = current_sum

    # ---------------------------------------------------------
    # 4. 短期生长动量特征 (仅 P2 到 P5 阶段可用，计算当前与上一阶段差值)
    # ---------------------------------------------------------
    for i in range(1, len(periods)):
        p_curr = periods[i]       # 当前阶段 (如 P2)
        p_prev = periods[i-1]     # 上一阶段 (如 P1)
        
        # 近期冠层扩张/衰老率 (P3 的正值代表返青，P5 的负值代表衰老)
        df[f'Delta_NDVI_{p_curr}'] = df[f'{p_curr}_NDVI'] - df[f'{p_prev}_NDVI']
        
        # 冠层水分流失率
        df[f'Delta_NDWI_{p_curr}'] = df[f'{p_curr}_NDWI'] - df[f'{p_prev}_NDWI']
        
        # 土壤墒情近期波动
        df[f'Delta_SM_{p_curr}'] = df[f'{p_curr}_SM'] - df[f'{p_prev}_SM']

    return df

def process_all_files(input_dir, output_dir):
    """
    遍历输入目录下的所有 CSV 文件，添加衍生特征后保存到输出目录。
    """
    # 确保输出目录存在
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"已创建输出文件夹: {output_dir}")

    # 获取所有年份的 CSV 文件
    # 假设文件名格式如 Merged_Features_2018.csv
    file_pattern = os.path.join(input_dir, "*.csv")
    csv_files = glob.glob(file_pattern)

    if not csv_files:
        print(f"在 {input_dir} 目录下未找到 CSV 文件。请检查路径。")
        return

    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        print(f"正在处理: {file_name} ...")
        
        # 读取原始数据
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"读取文件 {file_name} 失败: {e}")
            continue

        # 执行特征工程
        df_engineered = engineer_features(df)
        
        # 检查是否包含无穷大值 (由于除法产生)，将其替换为空值或0
        df_engineered.replace([np.inf, -np.inf], np.nan, inplace=True)
        # 用 0 或中位数填补由于分母极小导致的异常 nan，这里选择填充为 0 保障 RF 运行
        df_engineered.fillna(0, inplace=True)

        # 构建输出路径并保存
        output_path = os.path.join(output_dir, f"Engineered_{file_name}")
        df_engineered.to_csv(output_path, index=False)
        print(f"处理完成，已保存至: {output_path}")

if __name__ == "__main__":
    # --- 配置文件夹路径 ---
    # 如果你的脚本和 CSV 在同一个文件夹下，可以将 input_directory 设为 '.'
    # 或者指定为你存放 Merged_Features_2018.csv 等文件的绝对/相对路径
    input_directory = "./merged_dataset"      # 原始数据所在文件夹
    output_directory = "./engineered_features" # 包含衍生特征的新文件夹
    
    # 针对当前演示，如果你想直接在当前目录下运行处理当前目录的文件：
    # input_directory = "."
    # output_directory = "./output_features"
    
    process_all_files(input_directory, output_directory)
    print("所有年份数据处理完毕。")

已创建输出文件夹: ./engineered_features
正在处理: Merged_Features_2016.csv ...
处理完成，已保存至: ./engineered_features\Engineered_Merged_Features_2016.csv
正在处理: Merged_Features_2017.csv ...
处理完成，已保存至: ./engineered_features\Engineered_Merged_Features_2017.csv
正在处理: Merged_Features_2018.csv ...
处理完成，已保存至: ./engineered_features\Engineered_Merged_Features_2018.csv
正在处理: Merged_Features_2019.csv ...
处理完成，已保存至: ./engineered_features\Engineered_Merged_Features_2019.csv
正在处理: Merged_Features_2020.csv ...
处理完成，已保存至: ./engineered_features\Engineered_Merged_Features_2020.csv
正在处理: Merged_Features_2021.csv ...
处理完成，已保存至: ./engineered_features\Engineered_Merged_Features_2021.csv
所有年份数据处理完毕。
